In [ ]:
import tkinter as tk
import joblib
import numpy as np
import pandas as pd
import os
import sys
from tkinter import messagebox

# == Mengatur path supaya menjadi relative ==
# def resource_path(relative_path):
#     try:
#         # PyInstaller membuat folder temporary dan menyimpan path di _MEIPASS
#         base_path = sys._MEIPASS
#     except Exception:
#         base_path = os.path.abspath(".")

#     return os.path.join(base_path, relative_path)

# === Import Model ===

#model_path = resource_path("model_RFR.joblib")
model_path = "/content/drive/MyDrive/Colab Notebooks/model_RFR.joblib"
model = joblib.load(model_path) #Ubah model

# === Placeholder ===
def add_placeholder(entry, text):
    entry.insert(0, text)
    entry.config(fg="grey")

    def on_focus_in(event):
        if entry.get() == text:
            entry.delete(0, tk.END)
            entry.config(fg="black")

    def on_focus_out(event):
        if entry.get() == "":
            entry.insert(0, text)
            entry.config(fg="grey")

    entry.bind("<FocusIn>", on_focus_in)
    entry.bind("<FocusOut>", on_focus_out)


# === PARAMETER ===
parameter_names = [
    "KW Mill", "Inlet Mill Draught", "Outlet Mill Temperature",
    "DP Mill", "DP Bag Filter", "Mill Vibration",
    "Separator Power", "Blaine Cement", "Residue Cement (45 micron)"
]

display_names = [
    "KW Mill", "Inlet Mill Draught", "Outlet Mill Temp",
    "DP Mill", "DP Bag Filter", "Mill Vibration",
    "Separator Power", "Blaine Cement", "Residue Cement"
]

MIN_VAL = [0, -10, 0, 0, 0, 0, 0, 0, 0]
MAX_VAL = [6000, 0, 130, 50, 250, 3, 665, 7920, 43]

placeholder_text = [
    "0 - 6000", "-10 - 0", "0 - 130",
    "0 - 50", "0 - 250", "0 - 3",
    "0 - 665", "0 - 7920", "0 - 43"
]

# === FIXED VALUE FOR KW BF FAN AND KW SEPARATOR
# KW_BF_FAN = 200
# KW_SEPARATOR = 1630

# === Fungsi Predict ===
def predict(event=None):
    values = []

    for i, entry in enumerate(entries):
        raw = entry.get().strip()

        # Cek placeholder
        if raw == placeholder_text[i]:
            messagebox.showerror(
                "Input Kosong",
                f"{parameter_names[i]} belum diisi"
            )
            return

        # Ganti koma → titik
        raw = raw.replace(",", ".")

        # Validasi angka
        try:
            value = float(raw)
        except ValueError:
            messagebox.showerror(
                "Input Salah",
                f"{parameter_names[i]} harus berupa angka.\nGunakan titik (.) untuk desimal."
            )
            return

        # Validasi range (PER PARAMETER)
        if not (MIN_VAL[i] <= value <= MAX_VAL[i]):
            messagebox.showerror(
                "Di luar Range",
                f"{parameter_names[i]} harus di antara {MIN_VAL[i]} - {MAX_VAL[i]}"
            )
            return

        values.append(value)

    # values.append(KW_BF_FAN)
    # values.append(KW_SEPARATOR)

    # === MODEL ===
    data = pd.DataFrame(
        [values],
        columns=parameter_names
    )
    hasil = model.predict(data)
    hasil = hasil.astype(float)
    label_hasil.config(
        text=f"BF Fan Speed: {hasil[0,0]:.5f} \nFresh Feed: {hasil[0,1]:.5f} \nSeparator Speed: {hasil[0,2]:.5f}"
    )

# === GUI UTAMA ===
root = tk.Tk()
root.title("Demo GUI ML - Power Optimizing")
root.geometry("450x500")

# Enter = Predict
root.bind("<Return>", predict)

tk.Label(
    root,
    text="Simulasi Model",
    font=("Arial", 14, "bold")
).pack(pady=10)

frame_input = tk.Frame(root)
frame_input.pack()

entries = []
index = 0

for i in range(3):
    for j in range(3):
        param_frame = tk.Frame(frame_input)
        param_frame.grid(row=i, column=j, padx=10, pady=10)

        tk.Label(
            param_frame,
            text=display_names[index],
            font=("Arial", 9)
        ).pack()

        entry = tk.Entry(
            param_frame,
            width=10,
            justify="center"
        )
        entry.pack(pady=2)

        add_placeholder(
            entry,
            placeholder_text[index]
        )
        entries.append(entry)

        index += 1


tk.Button(
    root,
    text="PREDICT",
    font=("Arial", 12),
    bg="#00a86b",
    width=20,
    command=predict
).pack(pady=20)

label_hasil = tk.Label(
    root,
    text="Hasil Prediksi:",
    font=("Arial", 12))
label_hasil.pack()

root.mainloop()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


TclError: no display name and no $DISPLAY environment variable